# Fast tomo Procedure in Python 

In [94]:
# imports
import tomobase
import os
import ncempy
import stackview
import numpy as np
import pandas as pd 

directory = r'\\ematbyname\emat\TimC\IIT'
subdirectory = 'NSphereZIF-8'
ser_file = 'ftomo2_1.ser'
csv_file = 'angle_log2.csv'

imgs = []
for i in range(10000):
    try:
        imgs.append(ncempy.io.ser.fileSER(os.path.join(directory, subdirectory, ser_file)).getDataset(i)[0])
    except:
        break
    

data = np.stack(imgs, axis=0)
#stackview.slice(data)

df = pd.read_csv(os.path.join(directory, subdirectory, csv_file))
df.columns = df.columns.str.strip()
df = df.loc[:, ~df.columns.str.contains(r'^Unnamed')]

arr = df.iloc[:, :2].astype(float).to_numpy()
shape = data.shape[0]


t = np.linspace(arr[0, 0], arr[-1, 0], shape)

angles = np.zeros(shape, dtype=float)

angles[0] = arr[0, 1]
angles[-1] = arr[-1, 1]
for i in range(0, arr.shape[0]-1):
    mask = (t >= arr[i, 0]) & (t < arr[i+1, 0])
    angles[mask] = arr[i, 1]
    
print(angles.shape)
print(angles)

sino = tomobase.data.Sinogram(data, angles)


(174,)
[-70. -70. -70. -70. -68. -68. -68. -68. -66. -66. -66. -66. -64. -64.
 -64. -64. -62. -62. -62. -60. -60. -60. -60. -58. -58. -58. -58. -56.
 -56. -56. -56. -54. -54. -52. -52. -50. -50. -48. -48. -46. -46. -44.
 -44. -42. -42. -42. -40. -40. -38. -38. -36. -36. -34. -34. -32. -32.
 -30. -30. -28. -28. -26. -26. -24. -24. -22. -22. -20. -20. -18. -18.
 -16. -16. -14. -14. -12. -12. -10. -10.  -8.  -8.  -6.  -6.  -4.  -4.
  -2.  -2.   0.   0.   2.   2.   4.   4.   6.   6.   8.   8.  10.  10.
  12.  12.  14.  14.  16.  16.  18.  18.  20.  20.  22.  22.  24.  24.
  26.  26.  28.  28.  28.  30.  30.  32.  32.  34.  34.  36.  36.  38.
  38.  40.  40.  42.  42.  44.  44.  46.  46.  48.  48.  50.  50.  50.
  52.  52.  54.  54.  56.  56.  56.  58.  58.  58.  58.  60.  60.  60.
  60.  62.  62.  62.  62.  64.  64.  64.  64.  66.  66.  66.  68.  68.
  68.  68.  70.  70.  70.  70.]

[-70. -70. -70. -70. -68. -68. -68. -68. -66. -66. -66. -66. -64. -64.
 -64. -64. -62. -62. -62. -60. -60. -

In [95]:

sino = tomobase.processes.background_subtract_median(sino)
sino = tomobase.processes.align_sinogram_xcorr(sino)


2025-10-22 08:15:30,113 - DEBUG - ()
2025-10-22 08:15:30,114 - DEBUG - {'image': <tomobase.data.sinogram.Sinogram object at 0x0000017BD23AF090>}
2025-10-22 08:15:30,114 - DEBUG - {'image': <tomobase.data.sinogram.Sinogram object at 0x0000017BD23AF090>}
2025-10-22 08:15:31,587 - DEBUG - ()
2025-10-22 08:15:31,587 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000017BD23AF090>, 'shifts': None}
2025-10-22 08:15:31,587 - DEBUG - ()
2025-10-22 08:15:31,587 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000017BD23AF090>, 'shifts': None}
100%|██████████| 174/174 [00:00<00:00, 2350.41it/s]


In [87]:
stackview.crop(sino.data)

_Cropper(children=(HBox(children=(VBox(children=(VBox(children=(IntRangeSlider(value=(0, 174), description='Z'…

In [96]:
cropx = slice(287, 852)
cropy = slice(142, 829)
sino.data = sino.data[:, cropx, cropy]

stackview.slice(sino.data)



In [98]:
# import ssim
from skimage.metrics import structural_similarity as ssim

def measure_ssim(sino):
    measures = np.zeros_like(sino.angles, dtype=float)
    for i, a in enumerate(sino.angles):
        ssims = 0
        n_sinos = 0
        for j, b in enumerate(sino.angles):
            if a == b:
                n_sinos += 1
                if i != j:
                    val = ssim(sino.data[i, :, :], sino.data[j, :, :], data_range=sino.data.max()-sino.data.min())
                    ssims = max(val, ssims)
        measures[i] = ssims

    return measures

In [99]:
sino = tomobase.processes.align_sinogram_xcorr(sino)

ssims = measure_ssim(sino)
print(ssims)





2025-10-22 08:16:58,682 - DEBUG - ()
2025-10-22 08:16:58,682 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000017BD23AF090>, 'shifts': None}
2025-10-22 08:16:58,682 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000017BD23AF090>, 'shifts': None}
100%|██████████| 174/174 [00:00<00:00, 9440.28it/s]


[0.1981823  0.19952889 0.19952889 0.19457957 0.20072328 0.20072328
 0.19979801 0.18578201 0.19697802 0.19742504 0.19742504 0.18904573
 0.18847721 0.18789745 0.18847721 0.18201549 0.17999399 0.18151914
 0.18151914 0.17784672 0.17679154 0.17784672 0.16513185 0.1630464
 0.16633268 0.16633268 0.16606609 0.1704176  0.17134029 0.17134029
 0.17097941 0.17096292 0.17096292 0.17226223 0.17226223 0.17644187
 0.17644187 0.18098999 0.18098999 0.18403224 0.18403224 0.181652
 0.181652   0.18466864 0.18580551 0.18580551 0.19500846 0.19500846
 0.19519433 0.19519433 0.19455184 0.19455184 0.19105331 0.19105331
 0.19521994 0.19521994 0.20133333 0.20133333 0.20504071 0.20504071
 0.20319873 0.20319873 0.22355768 0.22355768 0.23139234 0.23139234
 0.23417462 0.23417462 0.24324298 0.24324298 0.25249121 0.25249121
 0.25614686 0.25614686 0.25846256 0.25846256 0.25988867 0.25988867
 0.26159524 0.26159524 0.26144397 0.26144397 0.26763327 0.26763327
 0.26798464 0.26798464 0.26736354 0.26736354 0.26841238 0.2684123

In [100]:
import copy
threshold = 0.26

# Sanity checks
assert ssims.shape[0] == sino.angles.shape[0], f"ssims length {ssims.shape} != angles {sino.angles.shape}"

# Compute groups by quantizing angles to small tolerance to avoid float jitter
# Use degrees tolerance of 1e-3 by default (adjust if your CSV uses different precision)
tol = 1e-3
bins = np.round(sino.angles / tol).astype(int)
unique_bins = np.unique(bins)

keep_mask = np.ones_like(bins, dtype=bool)
removed_count = 0

for b in unique_bins:
    idx = np.where(bins == b)[0]
    if idx.size <= 1:
        continue
    # loop: remove the lowest-ssim projection while there are >1 and the minimum is below threshold
    while idx.size > 1:
        svals = ssims[idx]
        minpos = np.nanargmin(svals)
        if svals[minpos] < threshold:
            rem = idx[minpos]
            keep_mask[rem] = False
            removed_count += 1
            # recompute idx
            idx = idx[idx != rem]
        else:
            break

print(f"Removed {removed_count} projections; kept {keep_mask.sum()} / {len(keep_mask)}")

# Build filtered sinogram
sino2 = copy.deepcopy(sino)
sino2.data = sino2.data[keep_mask, :, :]
sino2.angles = sino2.angles[keep_mask]

print(sino2.data.shape)
print(np.unique(sino2.angles, return_counts=True))


Removed 90 projections; kept 84 / 174

(84, 565, 687)
(array([-70., -68., -66., -64., -62., -60., -58., -56., -54., -52., -50.,
       -48., -46., -44., -42., -40., -38., -36., -34., -32., -30., -28.,
       -26., -24., -22., -20., -18., -16., -14., -12., -10.,  -8.,  -6.,
        -4.,  -2.,   0.,   2.,   4.,   6.,   8.,  10.,  12.,  14.,  16.,
        18.,  20.,  22.,  24.,  26.,  28.,  30.,  32.,  34.,  36.,  38.,
        40.,  42.,  44.,  46.,  48.,  50.,  52.,  54.,  56.,  58.,  60.,
        62.,  64.,  66.,  68.,  70.]), array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1]))
(84, 565, 687)
(array([-70., -68., -66., -64., -62., -60., -58., -56., -54., -52., -50.,
       -48., -46., -44., -42., -40., -38., -36., -34., -32., -30., -28.,
       -26., -24., -22., -20., -18., -16., -14., -12., -10.,  -

In [103]:
stackview.slice(sino2.data)

In [102]:
sino2.remove(slice(16,28))

In [104]:
sino=sino2
sino = tomobase.processes.align_tilt_axis_shift(sino)
stackview.slice(sino.data)

2025-10-22 08:18:29,309 - DEBUG - ()
2025-10-22 08:18:29,310 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000017BD11E5890>, 'method': 'fbp', 'offsets': 0.0}
  0%|          | 0/21 [00:00<?, ?it/s]2025-10-22 08:18:34,116 - DEBUG - ()
2025-10-22 08:18:34,117 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000017BE725F490>, 'method': 'fbp', 'iterations': 0, 'use_gpu': True}
2025-10-22 08:18:34,117 - INFO - Reconstructing...
2025-10-22 08:18:34,118 - INFO - Reconstruction using the FBP_CUDA algorithm on the GPU...
2025-10-22 08:18:34,116 - DEBUG - ()
2025-10-22 08:18:34,117 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000017BE725F490>, 'method': 'fbp', 'iterations': 0, 'use_gpu': True}
2025-10-22 08:18:34,117 - INFO - Reconstructing...
2025-10-22 08:18:34,118 - INFO - Reconstruction using the FBP_CUDA algorithm on the GPU...

100%|██████████| 565/565 [00:03<00:00, 146.19it/s]
2025-10-22 08:18:38,025 - INFO - type of volume: <class 'to

In [114]:
rec = tomobase.processes.reconstruct.optomo_reconstruct(sino, iterations=150)
rec.to_file(os.path.join(directory, subdirectory, 'result2.rec'))

2025-10-22 08:33:17,022 - DEBUG - ()
2025-10-22 08:33:17,022 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000017BD11E5890>, 'iterations': 150, 'use_gpu': True, 'weighted': False}
100%|██████████| 565/565 [28:30<00:00,  3.03s/it]
2025-10-22 09:01:47,555 - INFO - type of volume: <class 'tomobase.data.volume.Volume'>


In [115]:
#stackview.slice(sino.data)
#normalize sino.data
#sino.data = (sino.data - sino.data.min()) / (sino.data.max() - sino.data.min())
print(np.mean(sino.data))
stackview.orthogonal(rec.data)

0.10104495160676945


In [ ]:
sino2 = tomobase.processes.align_tilt_axis_rotation(sino, inplace=False)
stackview.slice(sino.data)

2025-10-22 08:07:34,481 - DEBUG - ()
2025-10-22 08:07:34,481 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000017BD12740D0>, 'method': 'fbp', 'angle': 0.0}
  0%|          | 0/9 [00:00<?, ?it/s]2025-10-22 08:07:36,505 - DEBUG - ()
2025-10-22 08:07:36,506 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000017BD1265410>, 'method': 'fbp', 'iterations': 0, 'use_gpu': True}
2025-10-22 08:07:36,506 - INFO - Reconstructing...
2025-10-22 08:07:36,506 - INFO - Reconstruction using the FBP_CUDA algorithm on the GPU...
2025-10-22 08:07:36,505 - DEBUG - ()
2025-10-22 08:07:36,506 - DEBUG - {'sino': <tomobase.data.sinogram.Sinogram object at 0x0000017BD1265410>, 'method': 'fbp', 'iterations': 0, 'use_gpu': True}
2025-10-22 08:07:36,506 - INFO - Reconstructing...
2025-10-22 08:07:36,506 - INFO - Reconstruction using the FBP_CUDA algorithm on the GPU...

100%|██████████| 565/565 [00:06<00:00, 86.36it/s]
2025-10-22 08:07:43,091 - INFO - type of volume: <class 'tomoba